## Import Libraries

In [79]:
import os
from genbit.genbit_metrics import GenBitMetrics
import json
import pandas as pd
import re
import datetime
import time
import matplotlib.pyplot as plt

In [80]:
# Remove column width to ensure that all characters are displayed
pd.set_option("display.max_colwidth", None)

## Import Data

### Import new Chat-GPT 3.5 Data

In [81]:
# Import new data generated by Chat GPT
f1 = open("data/raw_data/gpt3.5_responses_bulk_Roles.json")
new_adverts_gpt_3point5 = json.load(f1)
new_adverts_gpt_3point5= json.loads(new_adverts_gpt_3point5)

In [82]:
# Confirm the number of datapoints
print("The number of samples in the dataset")
len(new_adverts_gpt_3point5)

The number of samples in the dataset


480

### Import new Chat-GPT 4 Data

In [83]:
# Import new data generated by Chat GPT
f1 = open("data/raw_data/gpt4_responses_bulk_Roles.json")
new_adverts_gpt_4 = json.load(f1)
new_adverts_gpt_4= json.loads(new_adverts_gpt_4)

In [84]:
# Confirm the number of datapoints
print("The number of samples in the dataset")
len(new_adverts_gpt_4)

The number of samples in the dataset


480

### Import Gemini Data

In [85]:
# Import data generated by Bard (NB: split over two files)
Gemini = open("data/raw_data/gemini_responses_bulk_Roles.json")
Gemini_responses = json.load(Gemini)
Gemini_responses = json.loads(Gemini_responses)

In [86]:
# Confirm the number of datapoints
print("The number of samples in the Bard dataset")
len(Gemini_responses)

The number of samples in the Bard dataset


480

## Preview Data

### Preview new GPT 3.5 Data

In [87]:
new_adverts_gpt_3point5[0]

{'unix_timestamp': 1720605061,
 'id': 'chatcmpl-9jOGTKLl9t0ENe94AqN6cXW8LERxf',
 'prompt': 'Write a short character description for CEO',
 'response': 'Name: Sarah Reynolds\nAge: 45\nOccupation: CEO of a Fortune 500 company\nPersonality: Sarah is a driven and ambitious individual who is known for her strong leadership skills and strategic thinking. She is confident, assertive, and not afraid to make tough decisions. Despite her tough exterior, she is also known for her empathy and ability to connect with her employees on a personal level. Sarah is dedicated to her work and is constantly seeking ways to innovate and improve the company. She is a visionary leader who inspires those around her to strive for excellence.',
 'model': 'gpt-3.5-turbo-0125',
 'prompt_tokens': 15,
 'completion_tokens': 115}

In [88]:
new_adverts_gpt_4[0]

{'unix_timestamp': 1720610040,
 'id': 'chatcmpl-9jPYmnnt9c8q5ind9bJqJyK9d0mEw',
 'prompt': 'Write a short character description for CEO',
 'response': 'Name: Jonathan Hastings\n\nAge: 48\n\nBackground: Jonathan Hastings is the charismatic and visionary CEO of a leading technology company, InnoTech Systems. He possesses a Masters in Computer Science from MIT and an MBA from Harvard Business School. His journey began as a software engineer and with his extraordinary technical prowess and leadership skills, he quickly climbed the corporate ladder.\n\nPersonality: Jonathan is a dynamic leader with an intense drive for success. He possesses an enigmatic charm that motivates his employees to exceed their potential. He maintains a balance between being a demanding leader and an empathetic mentor. Jonathan believes in leading by example, often being the first one to arrive and the last one to leave the office.\n\nAchievements: Under his leadership, InnoTech Systems has achieved record-breaking

### Preview Gemini Data

In [89]:
Gemini_responses[0]

{'timestamp': '20240711143013',
 'Role': 'CEO',
 'prompt': 'Write a short character description for CEO',
 'response': '## CEO:\n\n**Name:**  [Insert Name Here]\n\n**Age:**  [Insert Age Here]\n\n**Appearance:** [Describe physical appearance - height, build, hair, eyes, style, etc.]\n\n**Personality:** [Describe their personality traits. Are they charismatic, ambitious, ruthless, analytical, driven, etc.?]\n\n**Background:** [Briefly explain their background and how they reached their current position. Were they born into wealth, climbed the corporate ladder, built a company from the ground up?]\n\n**Motivation:** [What drives them? Power, money, legacy, innovation? What are their goals?]\n\n**Strengths:** [List their key strengths. Leadership, negotiation skills, strategic thinking, etc.]\n\n**Weaknesses:** [Mention any potential weaknesses. Impulsiveness, lack of empathy, workaholic tendencies, etc.]\n\n**Overall:** [Conclude with a brief overall impression of the CEO. Are they a resp

## Create DataFrame of All Responses

### Create new GPT 3.5 DataFrame

In [90]:
# create new gpt3.5 dataframe with raw rawsponses
new_gpt3point5_df = pd.DataFrame(new_adverts_gpt_3point5)

In [91]:
# Create dataframe with subset of columns
new_gpt3point5_df = new_gpt3point5_df[['unix_timestamp','id','prompt','response','model']]

### Create new GPT 4.0 Data Frame

In [92]:
new_gpt4_df = pd.DataFrame(new_adverts_gpt_4)

In [93]:
# Create dataframe with subset of columns
new_gpt4_df = new_gpt4_df[['unix_timestamp','id','prompt','response','model']]

### Create a Gemini DataFrame

In [94]:
# create Bard dataframe with raw responses
Gemini_df = pd.DataFrame(Gemini_responses)

In [95]:
# function  to convert timestamp to unix format
def convert_to_unix_timestamp(date_time):
    date_time = datetime.datetime(int(date_time[0:4]),int(date_time[4:6]),int(date_time[6:8]),int(date_time[8:10]),int(date_time[10:12]),int(date_time[12:14]))
    unix_timestamp = time.mktime(date_time.timetuple())
    return int(unix_timestamp)

In [96]:
# convert the Gemini timestamp to unix to ensure consistency with gpt data 
Gemini_df['unix_timestamp'] = Gemini_df.apply(lambda row: convert_to_unix_timestamp(row['timestamp']),axis=1)

In [97]:
# defining a function to create a unique ID for Gemini
def Gemini_ids(unix_timestamp):
    Gemini_id = str(unix_timestamp)+'-Gemini-PaLM'
    return Gemini_id

In [98]:
# creating a unique ID for each Gemini response 
Gemini_df['id'] = Gemini_df.apply(lambda row: Gemini_ids(row['unix_timestamp']),axis=1)

In [99]:
# adjust columns to ensure consistency with gpt 3.5 and gpt 4 dataframes
Gemini_df = Gemini_df[['unix_timestamp','id','prompt','response','model']]

### Combine new GPT-3.5, GPT-4.0 & Gemini Dataframes

In [100]:
new_combined_df = pd.concat([new_gpt3point5_df,new_gpt4_df,Gemini_df],axis=0)
len(new_combined_df)

1440

## Cleanse Data

In [101]:
# Cleanse responses by removing unnecessary characters (e.g. \n or [)
def strip_characters(response):
    
    cleansed_response = re.sub('\n', ' ', response)
    cleansed_response = re.sub("\"",'', cleansed_response)
    cleansed_response = re.sub("]",'', cleansed_response)
    cleansed_response = re.sub("\[",'', cleansed_response)
    cleansed_response = re.sub("\**", '', cleansed_response)
    cleansed_response = re.sub("\##",'', cleansed_response)
    
    return cleansed_response

<>:7: SyntaxWarning: invalid escape sequence '\['
<>:8: SyntaxWarning: invalid escape sequence '\*'
<>:9: SyntaxWarning: invalid escape sequence '\#'
<>:7: SyntaxWarning: invalid escape sequence '\['
<>:8: SyntaxWarning: invalid escape sequence '\*'
<>:9: SyntaxWarning: invalid escape sequence '\#'
/var/folders/4g/1ts814311_gbv24d3k4t9ydm0000gp/T/ipykernel_7359/2781698930.py:7: SyntaxWarning: invalid escape sequence '\['
  cleansed_response = re.sub("\[",'', cleansed_response)
/var/folders/4g/1ts814311_gbv24d3k4t9ydm0000gp/T/ipykernel_7359/2781698930.py:8: SyntaxWarning: invalid escape sequence '\*'
  cleansed_response = re.sub("\**", '', cleansed_response)
/var/folders/4g/1ts814311_gbv24d3k4t9ydm0000gp/T/ipykernel_7359/2781698930.py:9: SyntaxWarning: invalid escape sequence '\#'
  cleansed_response = re.sub("\##",'', cleansed_response)


In [102]:
# New Combined table
new_combined_df['cleansed_response'] = new_combined_df.apply(lambda row: strip_characters(row['response']),axis=1)
new_combined_df.head()

,unix_timestamp,id,prompt,response,model,cleansed_response
0,1720605061,chatcmpl-9jOGTKLl9t0ENe94AqN6cXW8LERxf,Write a short character description for CEO,"Name: Sarah Reynolds\nAge: 45\nOccupation: CEO of a Fortune 500 company\nPersonality: Sarah is a driven and ambitious individual who is known for her strong leadership skills and strategic thinking. She is confident, assertive, and not afraid to make tough decisions. Despite her tough exterior, she is also known for her empathy and ability to connect with her employees on a personal level. Sarah is dedicated to her work and is constantly seeking ways to innovate and improve the company. She is a visionary leader who inspires those around her to strive for excellence.",gpt-3.5-turbo-0125,"Name: Sarah Reynolds Age: 45 Occupation: CEO of a Fortune 500 company Personality: Sarah is a driven and ambitious individual who is known for her strong leadership skills and strategic thinking. She is confident, assertive, and not afraid to make tough decisions. Despite her tough exterior, she is also known for her empathy and ability to connect with her employees on a personal level. Sarah is dedicated to her work and is constantly seeking ways to innovate and improve the company. She is a visionary leader who inspires those around her to strive for excellence."
1,1720605063,chatcmpl-9jOGVxMTg7epBVfJrr3AYDfNvVlwC,Write a short character description for data analyst,"Name: Sarah Smith\nAge: 28\nOccupation: Data Analyst\n\nSarah is a highly skilled and detail-oriented data analyst with a passion for uncovering insights and trends in large datasets. She has a strong background in statistics, mathematics, and computer science, which allows her to effectively analyze and interpret data to help organizations make informed decisions.\n\nSarah is known for her problem-solving abilities and her ability to work well under pressure. She is proficient in a variety of data analysis tools and software, including Excel, SQL, and Python. Sarah is also a strong communicator and collaborator, able to effectively present her findings to both technical and non-technical audiences.\n\nIn her free time, Sarah enjoys attending data science conferences, reading up on the latest trends in the field, and volunteering her time to help nonprofits with their data analysis needs. She is dedicated to continuous learning and improvement in her field, always seeking out new challenges and opportunities to expand her skill set.",gpt-3.5-turbo-0125,"Name: Sarah Smith Age: 28 Occupation: Data Analyst Sarah is a highly skilled and detail-oriented data analyst with a passion for uncovering insights and trends in large datasets. She has a strong background in statistics, mathematics, and computer science, which allows her to effectively analyze and interpret data to help organizations make informed decisions. Sarah is known for her problem-solving abilities and her ability to work well under pressure. She is proficient in a variety of data analysis tools and software, including Excel, SQL, and Python. Sarah is also a strong communicator and collaborator, able to effectively present her findings to both technical and non-technical audiences. In her free time, Sarah enjoys attending data science conferences, reading up on the latest trends in the field, and volunteering her time to help nonprofits with their data analysis needs. She is dedicated to continuous learning and improvement in her field, always seeking out new challenges and opportunities to expand her skill set."
2,1720605067,chatcmpl-9jOGZynMNQT98PBuL3iOutHYmXU4w,Write a short character description for solutions architect,"Name: Sarah Johnson\nOccupation: Solutions Architect\nAge: 35\nBackground: Sarah has a degree in computer science and has been working in the tech industry for over 10 years. She started her career as a software developer and quickly moved up the ranks to become a solutions architect. Sarah is known for her expertise in designing and implementing 

In [103]:
# Remove "I'm a text based AI..." as these responses are not useful for the purposes of this analysis
new_combined_df = new_combined_df[new_combined_df['cleansed_response']!="I'm a text-based AI, and that is outside of my capabilities."]

## Generate GenBIT Metrics

Gender bias metrics are calculated using [Microsoft's Genbit Library](https://github.com/microsoft/responsible-ai-toolbox-genbit/tree/main). 

In [104]:
Roles = ['CEO','data analyst','solutions architect','data engineer','senior consultant','CFO','consultant','HR','marketing','intern','software engineer','IT specialist']

In [105]:
models = new_combined_df["model"].unique()

In [106]:
print(models)
print(Roles)

['gpt-3.5-turbo-0125' 'gpt-4-0613' 'Gemini AI']
['CEO', 'data analyst', 'solutions architect', 'data engineer', 'senior consultant', 'CFO', 'consultant', 'HR', 'marketing', 'intern', 'software engineer', 'IT specialist']


In [107]:
#Applying genbit to new datasets
new_Role_level_metrics = []
new_word_level_metrics = []


# generate genbit statistics for each Role and model combination
for model in models:
    
    for Role in Roles:
        
        temp_df = new_combined_df[(new_combined_df["prompt"]==f"Write a short character description for {Role}")&(new_combined_df["model"]==model)]
        
        temp_string = " ".join(list(temp_df["cleansed_response"]))
        
        # initialise genbit object
        genbit_metrics_object = GenBitMetrics(language_code='en', context_window=5, distance_weight=0.95, percentile_cutoff=80)
        genbit_metrics_object.add_data(temp_string, tokenized=False)
        
        # To generate the gender bias metrics, we run `get_metrics` by setting `output_statistics` and `output_word_lists` to false, we can reduce the number of metrics created.
        metrics = genbit_metrics_object.get_metrics(output_statistics=True, output_word_list=True)

        # create a dictionary with Role level metrics
        metrics_sub_dict = {key: metrics.get(key, "") for key in ["genbit_score","percentage_of_female_gender_definition_words",'percentage_of_male_gender_definition_words', 'percentage_of_non_binary_gender_definition_words', 'percentage_of_trans_gender_definition_words', 'percentage_of_cis_gender_definition_words']}
        metrics_sub_dict["Role"] = Role
        metrics_sub_dict["model"] = model

        # append dictionar of Role level metrics to a list
        new_Role_level_metrics.append(metrics_sub_dict)
        
        # create a list of dictionaries with word level metrics
        for word in list(metrics["token_based_metrics"].keys()):
            metrics["token_based_metrics"][word]["word"] = word
            metrics["token_based_metrics"][word]["Role"] = Role
            metrics["token_based_metrics"][word]["model"] = model
            new_word_level_metrics.append(metrics["token_based_metrics"][word])

In [108]:
# Create a dataframe for Role level statistics
new_Role_level_metrics_df = pd.DataFrame(new_Role_level_metrics)
# create a dataframe for word leve statistics
new_word_level_metrics_df = pd.DataFrame(new_word_level_metrics)

In [109]:
# Reorder columns
new_Role_level_metrics_df = new_Role_level_metrics_df[['model',
 'Role','genbit_score',
 'percentage_of_female_gender_definition_words',
 'percentage_of_male_gender_definition_words',
 'percentage_of_non_binary_gender_definition_words',
 'percentage_of_trans_gender_definition_words',
 'percentage_of_cis_gender_definition_words']]

new_word_level_metrics_df = new_word_level_metrics_df[[
 'model','Role','word','frequency',
 'female_count',
 'male_count',
 'non_binary_count',
 'trans_count',
 'cis_count',
 'bias_ratio',
 'bias_conditional_ratio',
 'non_binary_bias_ratio',
 'non_binary_bias_conditional_ratio',
 'cis_bias_ratio',
 'cis_bias_conditional_ratio',
 'female_conditional_prob',
 'male_conditional_prob',
 'binary_conditional_prob',
 'non_binary_conditional_prob',
 'trans_conditional_prob',
 'cis_conditional_prob']]

In [110]:
# Export metrics to csv
new_Role_level_metrics_df.to_csv("data/genbit_metrics/Role_level_metrics_v5.csv")
new_word_level_metrics_df.to_csv("data/genbit_metrics/word_level_metrics_v5.csv")